In [11]:
import torch
import torch.nn as nn
import numpy as np
import onnx
import onnxruntime as ort
import tensorflow as tf
import os

In [12]:
# SimpleLinearNetwork (PyTorch)
class SimpleLinearNetworkPT(nn.Module):
    def __init__(self):
        super(SimpleLinearNetworkPT, self).__init__()
        self.linear = nn.Linear(32, 32)

    def forward(self, data) -> torch.Tensor:
        return self.linear(data)

# SimpleLinearNetwork (TensorFlow)
class SimpleLinearNetworkTF(tf.keras.Model):
    def __init__(self):
        super(SimpleLinearNetworkTF, self).__init__()
        self.dense = tf.keras.layers.Dense(32)

    def call(self, inputs):
        return self.dense(inputs)

In [13]:
data_torch = torch.randn(1, 32)
data_tf = tf.random.normal([1, 32])

# PyTorch
model_torch = SimpleLinearNetworkPT()
model_torch.eval()
model_torch(data_torch)

# TensorFlow
model_tf = SimpleLinearNetworkTF()
model_tf.call(data_tf)

<tf.Tensor: shape=(1, 32), dtype=float32, numpy=
array([[ 1.5138191 ,  0.13083646,  1.7615726 ,  0.37347957,  0.69801646,
         0.5699756 , -1.8353647 ,  2.4928472 ,  0.26831347, -1.0293067 ,
         1.7667781 , -1.09649   ,  0.24848941, -0.2505804 ,  1.0136628 ,
         4.0145187 ,  0.8510701 ,  0.1984998 ,  0.45782477,  1.0481912 ,
        -2.6235588 , -0.7694062 ,  0.49637988, -1.5896652 , -0.25930178,
        -2.9103858 , -1.9329911 , -0.2045173 ,  0.4414521 ,  0.5980465 ,
         2.6357424 ,  1.5148909 ]], dtype=float32)>

In [14]:
# Export PyTorch model
scripted_net = torch.jit.trace(model_torch, data_torch)
scripted_net.save("simple_linear_network.pt")

In [15]:
# Export to ONNX
torch.onnx.export(
    model_torch,
    data_torch,
    "simple_linear_network.onnx",
    input_names=["data"],
    output_names=["processed_data"],
    opset_version=11
)

In [20]:
# Export to TensorFlow Lite
data_shape = data_tf.shape
concrete_func = tf.function(model_tf).get_concrete_function(tf.TensorSpec(data_shape, data_tf.dtype))
converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func], model_tf)
tflite_model = converter.convert()
with open("simple_linear_network.tflite", "wb") as f:
    f.write(tflite_model)

I0000 00:00:1736811520.318754  294862 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1736811520.318816  294862 single_machine.cc:361] Starting new session
W0000 00:00:1736811520.325046  294862 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1736811520.325059  294862 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2025-01-14 00:38:40.334823: I tensorflow/compiler/mlir/lite/flatbuffer_export.cc:3893] Estimated count of arithmetic ops: 2048  ops, equivalently 1024  MACs
